# Streaming

LangGraph implements a streaming system to surface real-time updates. Streaming is crucial for enhancing the responsiveness of applications built on LLMs. By displaying output progressively, even before a complete response is ready, streaming significantly improves user experience (UX), particularly when dealing with the latency of LLMs

In [36]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## Basic Usage

LangGraph graphs expose the `stream` (sync) and `astream` (async) methods to yield streamed outputs as iterators. Pass one or more `stream modes` to control what data you receive.

```python
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom"],
    version="v2",
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Status: {chunk['data']['status']}")
```

Output:

```python
Status: thinking of a joke...
Node generate_joke updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()
    writer({"status": "Thinking about a joke..."})
    return {
        "joke": f"Why did the {state['topic']} go to school? To get a sundae education!"
    }


graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

# version="v2" enables streaming
for chunk in graph.stream(
    {"topic": "ice cream"}, stream_mode=["updates", "custom"], version="v2"
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated:{state}")
    elif chunk["type"] == "custom":
        print(f"Status: {chunk["data"]["status"]}")

print("=================\n======================")


# version="v1" enables streaming
for chunk in graph.stream(
    {"topic": "ice cream"}, stream_mode=["updates", "custom"], version="v1"
):
    print(chunk)


print("=================\n======================")

for part in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        # ValuesStreamPart — full state snapshot after each step
        print(f"State: topic={part['data']['topic']}")
    elif part["type"] == "updates":
        # UpdatesStreamPart — only the changed keys from each node
        for node_name, state in part["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif part["type"] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = part["data"]
        print(msg.content, end="", flush=True)
    # elif part["type"] == "custom":
    #     # CustomStreamPart — arbitrary data from get_stream_writer()
    #     print(f"Progress: {part['data']['progress']}%")

Status: Thinking about a joke...
Node generate_joke updated:{'joke': 'Why did the ice cream go to school? To get a sundae education!'}
('custom', {'status': 'Thinking about a joke...'})
('updates', {'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}})
State: topic=ice cream
Node `generate_joke` updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}
State: topic=ice cream


## Stream modes

Pass one or more of the following stream modes as a list to the `stream` or `astream` methods:

|Mode|	Type|	Description|
|---|---|---|
|values|	ValuesStreamPart|	Full state after each step.|
|updates|	UpdatesStreamPart|	State updates after each step. Multiple updates in the same step are streamed separately.|
|messages|	MessagesStreamPart|	2-tuples of (LLM token, metadata) from LLM calls.|
|custom|	CustomStreamPart|	Custom data emitted from nodes via get_stream_writer.|
|checkpoints|	CheckpointStreamPart|	Checkpoint events (same format as get_state()). Requires a checkpointer.|
|tasks|	TasksStreamPart|	Task start/finish events with results and errors. Requires a checkpointer.|
|debug|	DebugStreamPart	|All available info — combines checkpoints and tasks with extra metadata.|

## Graph state

Use the stream modes `updates` and `values` to stream the state of the graph as it executes.

* updates streams the **updates** to the state after each step of the graph.
* values streams the **full value** of the state after each step of the graph.

In [19]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    topic: str
    joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}


graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

In [21]:
# Use this to stream only the state updates returned by the nodes after each step. The streamed outputs include
# the name of the node as well as the update.

for chunk in graph.stream(
    {"topic": "ice cream"}, stream_mode=["updates"], version="v2"
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated: {state}")

Node refine_topic updated: {'topic': 'ice cream and cats'}
Node generate_joke updated: {'joke': 'This is a joke about ice cream and cats'}


In [32]:
# Use this to stream the full state of the graph after each step.

for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",
    version="v2",
):
    if chunk["type"] == "values":
        print(f"topic: {chunk['data']['topic']},")

topic: ice cream,
topic: ice cream and cats,
topic: ice cream and cats,


## LLM tokens

Use the messages streaming mode to stream Large Language Model (LLM) outputs **token by token** from any part of your graph, including nodes, tools, subgraphs, or tasks.

The streamed output from `messages` mode is a tuple `(message_chunk, metadata)` where:

* message_chunk: the token or message segment from the LLM.
* metadata: a dictionary containing details about the graph node and LLM invocation.

In [34]:
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END


@dataclass
class MyState:
    topic: str
    joke: str = ""


model = init_chat_model("claude-haiku-4-5", temperature=0, max_tokens=100)


def call_model(state: MyState):
    """Call thr LLM to generate a joke about a topic"""

    model_response = model.invoke(
        [
            {"role": "user", "content": f"Generate a joke about {state.topic}"},
        ]
    )

    return {"joke": model_response.content}


graph = (
    StateGraph(MyState)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .add_edge("call_model", END)
    .compile()
)

# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format

for chunk in graph.stream(
    {"topic": "ice cream"}, stream_mode=["messages"], version="v2"
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        if message_chunk.content:
            print(message_chunk.content, end="|", flush=True)

Why| did the ice cream go| to the| doctor?|

Because it was| feeling a| little *|mel|anch|oly*!| 🍦|

## Filter by LLM invocation

You can associate `tags` with LLM invocations to filter the streamed tokens by LLM invocation.

In [39]:
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import START, StateGraph

# The joke_model is tagged with "joke"
joke_model = init_chat_model(model="claude-haiku-4-5", tags=["joke"])
# The poem_model is tagged with "poem"
poem_model = init_chat_model(model="gpt-4.1-mini", tags=["poem"])


class State(TypedDict):
    topic: str
    joke: str
    poem: str


async def call_model(state, config):
    topic = state["topic"]
    print("Writing joke...")
    # Note: Passing the config through explicitly is required for python < 3.11
    # Since context var support wasn't added before then: https://docs.python.org/3/library/asyncio-task.html#creating-tasks
    # The config is passed through explicitly to ensure the context vars are propagated correctly
    # This is required for Python < 3.11 when using async code. Please see the async section for more details
    joke_response = await joke_model.ainvoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}],
        config,
    )
    print("\n\nWriting poem...")
    poem_response = await poem_model.ainvoke(
        [{"role": "user", "content": f"Write a short poem about {topic}"}],
        config,
    )
    return {"joke": joke_response.content, "poem": poem_response.content}


graph = StateGraph(State).add_node(call_model).add_edge(START, "call_model").compile()

# The stream_mode is set to "messages" to stream LLM tokens
# The metadata contains information about the LLM invocation, including the tags
async for chunk in graph.astream(
    {"topic": "cats"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        if metadata["tags"] == ["joke"]:
            print(msg.content, end="|", flush=True)
        elif metadata["tags"] == ["poem"]:
            print(msg.content, end="|", flush=True)

Writing joke...
|Why| don't cats play| poker in| the| jungle?

Because there| are| too| many ch|eetahs!||

Writing poem...
|Soft| paws| tread| on| silent| nights|,|  
|Wh|isk|ers| twitch| in| moon|lit| lights|.|  
|Eyes| that| gle|am| with| secrets| deep|,|  
|In| cozy| laps|,| they| p|urr| and| sleep|.|  

|Myst|ery| wrapped| in| fur| so| fine|,|  
|A| gentle| heart|,| a| soul| divine|.|  
|Cats|,| the| shadows|’| quiet| art|,|  
|Forever| wild| within| our| heart|.||||

## Filter by node

To stream tokens only from specific nodes, use `stream_mode="messages"` and filter the outputs by the `langgraph_node` field in the streamed metadata:

In [41]:
from typing import TypedDict
from langgraph.graph import START, StateGraph
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=40)


class State(TypedDict):
    topic: str
    joke: str
    poem: str


def write_joke(state: State):
    topic = state["topic"]
    joke_response = model.invoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}]
    )
    return {"joke": joke_response.content}


def write_poem(state: State):
    topic = state["topic"]
    poem_response = model.invoke(
        [{"role": "user", "content": f"Write a short poem about {topic}"}]
    )
    return {"poem": poem_response.content}


graph = (
    StateGraph(State)
    .add_node(write_joke)
    .add_node(write_poem)
    # write both the joke and the poem concurrently
    .add_edge(START, "write_joke")
    .add_edge(START, "write_poem")
    .compile()
)

# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    {"topic": "cats"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the langgraph_node field in the metadata
        # to only include the tokens from the write_poem node
        if msg.content and metadata["langgraph_node"] == "write_poem":
            print(msg.content, end="|", flush=True)
        elif msg.content and metadata["langgraph_node"] == "write_joke":
            print(msg.content, end="|", flush=True)

Why| did| the| cat| sit| on| the| computer|?

|Because| it| wanted| to| keep| an| eye| on| the| mouse|!|In| moon|lit| grace| they| softly| tread|,|  
|Wh|isk|ers| twitch| and| tails| are| spread|.|  
|Silent| paws| on| midnight| floors|,|  
|Myst|ery| behind| closed| doors|.|  

|Eyes| that| gle|am| like| stars| above|

## Custom data

To send **custom user-defined data** from inside a LangGraph node or tool, follow these steps:

1. Use `get_stream_writer` to access the stream writer and emit custom data.
2. Set `stream_mode="custom"` when calling .`stream`() or .`astream`() to get the custom data in the stream. You can combine multiple modes (e.g., ["`updates`", "`custom`"]), but at least one must be "`custom`".

In [42]:
from typing import TypedDict
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START


class State(TypedDict):
    query: str
    answer: str


def node(state: State):
    # Get the stream writer to send custom data
    writer = get_stream_writer()
    # Emit a custom key-value pair (e.g., progress update)
    writer({"custom_key": "Generating custom data inside node"})
    return {"answer": "some data"}


graph = StateGraph(State).add_node(node).add_edge(START, "node").compile()

inputs = {"query": "example"}

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    if chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']['custom_key']}")

Custom event: Generating custom data inside node


In [48]:
from langchain.tools import tool
from langgraph.config import get_stream_writer


@tool
def query_database(query: str) -> dict:
    """Query the database."""
    # Access the stream writer to send custom data
    writer = get_stream_writer()
    # Emit a custom key-value pair (e.g., progress update)
    writer({"data": "Retrieved 0/100 records", "type": "progress"})
    # perform query
    # Emit another custom key-value pair
    writer({"data": "Retrieved 100/100 records", "type": "progress"})
    return {"some_key": "some-answer"}


graph = (
    StateGraph(State)
    .add_node(query_database)
    .add_edge(START, "query_database")
    .compile()
)

inputs = {"query": "example"}

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    if chunk["type"] == "custom":
        print(f"{chunk['data']['type']}: {chunk['data']['data']}")

progress: Retrieved 0/100 records
progress: Retrieved 100/100 records


## Subgraph outputs

To include outputs from subgraphs in the streamed outputs, you can set `subgraphs=True` in the .`stream`() method of the parent graph. This will stream outputs from both the parent graph and any subgraphs.

The outputs will be streamed as tuples (`namespace, data`), where `namespace` is a tuple with the path to the node where a subgraph is invoked, e.g. `("parent_node:<task_id>", "child_node:<task_id>")`.

In [49]:
from langgraph.graph import START, StateGraph
from typing import TypedDict


# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # note that this key is shared with the parent graph state
    bar: str


def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}


def subgraph_node_2(state: SubgraphState):
    return {"foo": state["foo"] + state["bar"]}


subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()


# Define parent graph
class ParentState(TypedDict):
    foo: str


def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}


builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,
    version="v2",
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            print(f"Subgraph {chunk['ns']}: {chunk['data']}")
        else:
            print(f"Root: {chunk['data']}")

Root: {'node_1': {'foo': 'hi! foo'}}
Subgraph ('node_2:b2aee919-2004-677a-2429-38d3b55bf114',): {'subgraph_node_1': {'bar': 'bar'}}
Subgraph ('node_2:b2aee919-2004-677a-2429-38d3b55bf114',): {'subgraph_node_2': {'foo': 'hi! foobar'}}
Root: {'node_2': {'foo': 'hi! foobar'}}


## Checkpoints

Use the `checkpoints` streaming mode to receive checkpoint events as the graph executes. Each checkpoint event has the same format as the output of `get_state`(). Requires a `checkpointer`.

In [52]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    topic: str
    joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}


from langgraph.checkpoint.memory import MemorySaver

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="checkpoints",
    version="v2",
):
    if chunk["type"] == "checkpoints":
        print(chunk["data"])

{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d02-afa8-6039-bfff-52cd27b2e4c4'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '87ca620a-479f-3b8f-f935-39fd2f2a0231', 'name': '__start__', 'interrupts': (), 'state': None}]}
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d02-afa9-6cf8-8000-e75235e6333c'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d02-afa8-6039-bfff-52cd27b2e4c4'}}, 'values': {'topic': 'ice cream'}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['refine_topic'], 'tasks': [{'id': '2c45c1ca-4328-deb6-246f-cd23944c03dd', 'name': 'refine_topic', 'interrupts': (), 'state': None}]}
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d02-afac-644c-8001-02dca0f64276'}}, 'parent_config': {'

## Tasks

Use the `tasks` streaming mode to receive task start and finish events as the graph executes. Task events include information about which node is running, its results, and any errors. Requires a `checkpointer`.

In [53]:
from langgraph.checkpoint.memory import MemorySaver

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="tasks",
    version="v2",
):
    if chunk["type"] == "tasks":
        print(chunk["data"])

{'id': 'ba21987e-aaf8-3d51-c415-151c25de270f', 'name': 'refine_topic', 'input': {'topic': 'ice cream'}, 'triggers': ('branch:to:refine_topic',)}
{'id': 'ba21987e-aaf8-3d51-c415-151c25de270f', 'name': 'refine_topic', 'error': None, 'result': {'topic': 'ice cream and cats'}, 'interrupts': []}
{'id': '2ad6aeb8-ff7b-279e-dce5-6a10698d833c', 'name': 'generate_joke', 'input': {'topic': 'ice cream and cats'}, 'triggers': ('branch:to:generate_joke',)}
{'id': '2ad6aeb8-ff7b-279e-dce5-6a10698d833c', 'name': 'generate_joke', 'error': None, 'result': {'joke': 'This is a joke about ice cream and cats'}, 'interrupts': []}


## Debug

Use the `debug` streaming mode to stream as much information as possible throughout the execution of the graph. The streamed outputs include the name of the node as well as the full state.

In [57]:
from langgraph.checkpoint.memory import MemorySaver

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="debug",
    version="v2",
):
    if chunk["type"] == "debug":
        print(chunk["data"])

{'step': -1, 'timestamp': '2026-03-17T07:11:46.034065+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d08-ef1f-60bf-bfff-d79c1b4eff48'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'c6ce34b1-d2ef-62c0-fd20-be86f82cee91', 'name': '__start__', 'interrupts': (), 'state': None}]}}
{'step': 0, 'timestamp': '2026-03-17T07:11:46.035074+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d08-ef21-67b1-8000-6ec65c9b7fd8'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d08-ef1f-60bf-bfff-d79c1b4eff48'}}, 'values': {'topic': 'ice cream'}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['refine_topic'], 'tasks': [{'id': 'd07152fd-2b3d-d4f6-08e3-d22d60e515c0', 'name': 'refine

## Multiple modes at once

You can pass a list as the `stream_mode` parameter to stream multiple modes at once.
With version="v2", every chunk is a `StreamPart` dict. Use chunk["type"] to distinguish between modes:

In [68]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    topic: str
    joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}


from langgraph.checkpoint.memory import MemorySaver

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

inputs = {"topic": "ice cream"}

for chunk in graph.stream(
    inputs,
    stream_mode=[
        "updates",
        "custom",
        "values",
        "messages",
        "debug",
        "checkpoints",
        "tasks",
        "root",
        "error",
    ],
    version="v2",
    config=config,
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']}")
    elif chunk["type"] == "values":
        print(f"State: {chunk['data']}")
    elif chunk["type"] == "messages":
        msg, metadata = chunk["data"]
    elif chunk["type"] == "debug":
        print(f"Debug: {chunk['data']}")
    elif chunk["type"] == "checkpoints":
        print(f"Checkpoints: {chunk['data']}")
    elif chunk["type"] == "tasks":
        print(f"Tasks: {chunk['data']}")
    elif chunk["type"] == "root":
        print(f"Root: {chunk['data']}")
    elif chunk["type"] == "error":
        print(f"Error: {chunk['data']}")

Checkpoints: {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d14-6b02-63bb-bfff-04c054ac2b90'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '614f4157-7bf6-7dcb-754b-f30952ce43ba', 'name': '__start__', 'interrupts': (), 'state': None}]}
Debug: {'step': -1, 'timestamp': '2026-03-17T07:16:54.303619+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d14-6b02-63bb-bfff-04c054ac2b90'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '614f4157-7bf6-7dcb-754b-f30952ce43ba', 'name': '__start__', 'interrupts': (), 'state': None}]}}
State: {'topic': 'ice cream'}
Checkpoints: {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f121d14-6b08-613f-8000-0acb2cd8311

# Advanced
## Use with any LLM

You can use `stream_mode="custom"` to stream data from **any LLM API**—even if that API does **not** implement the LangChain chat model interface.

This lets you integrate raw LLM clients or external services that provide their own streaming interfaces, making LangGraph highly flexible for custom setups.

In [ ]:
import operator
import json

from typing import TypedDict
from typing_extensions import Annotated
from langgraph.graph import StateGraph, START

from openai import AsyncOpenAI

openai_client = AsyncOpenAI()
model_name = "gpt-4.1-mini"


async def stream_tokens(model_name: str, messages: list[dict]):
    response = await openai_client.chat.completions.create(
        messages=messages, model=model_name, stream=True, max_tokens=40
    )
    role = None
    async for chunk in response:
        delta = chunk.choices[0].delta

        if delta.role is not None:
            role = delta.role

        if delta.content:
            yield {"role": role, "content": delta.content}


# this is our tool
async def get_items(place: str) -> str:
    """Use this tool to list items one might find in a place you're asked about."""
    writer = get_stream_writer()
    response = ""
    async for msg_chunk in stream_tokens(
        model_name,
        [
            {
                "role": "user",
                "content": (
                    "Can you tell me what kind of items "
                    f"i might find in the following place: '{place}'. "
                    "List at least 3 such items separating them by a comma. "
                    "And include a brief description of each item."
                ),
            }
        ],
    ):
        response += msg_chunk["content"]
        writer(msg_chunk)

    return response


class State(TypedDict):
    messages: Annotated[list[dict], operator.add]


# this is the tool-calling graph node
async def call_tool(state: State):
    ai_message = state["messages"][-1]
    tool_call = ai_message["tool_calls"][-1]

    function_name = tool_call["function"]["name"]
    if function_name != "get_items":
        raise ValueError(f"Tool {function_name} not supported")

    function_arguments = tool_call["function"]["arguments"]
    arguments = json.loads(function_arguments)

    function_response = await get_items(**arguments)
    tool_message = {
        "tool_call_id": tool_call["id"],
        "role": "tool",
        "name": function_name,
        "content": function_response,
    }
    return {"messages": [tool_message]}


graph = StateGraph(State).add_node(call_tool).add_edge(START, "call_tool").compile()

In [71]:
inputs = {
    "messages": [
        {
            "content": None,
            "role": "assistant",
            "tool_calls": [
                {
                    "id": "1",
                    "function": {
                        "arguments": '{"place":"bedroom"}',
                        "name": "get_items",
                    },
                    "type": "function",
                }
            ],
        }
    ]
}

async for chunk in graph.astream(
    inputs,
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        print(chunk["data"]["content"], end="|", flush=True)

In| a| bedroom|,| you| might| find|:

|1|.| Bed| –| A| piece| of| furniture| used| for| sleeping| or| resting|,| typically| consisting| of| a| mattress| supported| by| a| frame|.
|2|.| D|resser| –| A| storage| unit| with| drawers| used| for| organizing| clothes| and| personal| items|.
|3|.| Night|stand| –| A| small| table| placed| beside| the| bed|,| often| used| to| hold| a| lamp|,| alarm| clock|,| or| personal| belongings|.|